# PAS Diagnostics — §4.1, §4.2, §4.3 Blocker Checks
## Using Actual Column Names from Your CSV

Your CSV has already been scored with:
- `composite_score` — final risk score (1-10)
- `adequacy_score`, `capacity_score`, `appetite_score`, `environment_score` — components

This notebook checks the three blockers using your actual column structure.

## Cell 1: Load Data

In [1]:
import pandas as pd
import numpy as np

csv_path = r'C:\Box\Box\BOX Subhashree Singh\Business\PAS\physician_scores_20260429_141959.csv'

print("Loading CSV...")
df = pd.read_csv(csv_path, low_memory=False)

print(f"✅ Loaded {len(df):,} physicians")
print(f"✅ {len(df.columns)} columns")
print(f"\nKey columns found:")
print(f"  - composite_score: {df['composite_score'].describe()}")

Loading CSV...
✅ Loaded 165,623 physicians
✅ 41 columns

Key columns found:
  - composite_score: count    165623.000000
mean          4.483085
std           1.026577
min           2.000000
25%           4.000000
50%           4.000000
75%           5.000000
max           8.000000
Name: composite_score, dtype: float64


## Cell 2: §4.1 BLOCKER — Adequacy Reality Check

**Question:** Is Adequacy (40% of composite) measuring real loss experience?

**Check:** 
- What % of physicians have `kpi_indemnity_frequency = 0` (zero claims)?
- What % have `adequacy_score = 0`?
- Do they match?

In [2]:
print("="*80)
print("§4.1 BLOCKER — ADEQUACY REALITY CHECK")
print("="*80)
print()

# Check zero claims
zero_claims = (df['kpi_total_frequency'] == 0).sum()
pct_zero_claims = (zero_claims / len(df)) * 100

print(f"Physicians with ZERO claims (kpi_total_frequency = 0):")
print(f"  Count: {zero_claims:,}")
print(f"  Percentage: {pct_zero_claims:.1f}%")

# Check adequacy score distribution
print(f"\nAdequacy Score Distribution:")
print(f"  Mean: {df['adequacy_score'].mean():.3f}")
print(f"  Median: {df['adequacy_score'].median():.3f}")
print(f"  Min: {df['adequacy_score'].min():.3f}")
print(f"  Max: {df['adequacy_score'].max():.3f}")
print(f"  Std Dev: {df['adequacy_score'].std():.3f}")

# Count zeros
zero_adequacy = (df['adequacy_score'] == 0).sum()
pct_zero_adequacy = (zero_adequacy / len(df)) * 100

print(f"\nPhysicians with Adequacy Score = 0:")
print(f"  Count: {zero_adequacy:,}")
print(f"  Percentage: {pct_zero_adequacy:.1f}%")

# Check the real issue: loss cost vs loss ratio
print(f"\nLoss Cost Distribution (kpi_total_loss_cost):")
print(f"  Mean: {df['kpi_total_loss_cost'].mean():.2f}")
print(f"  Median: {df['kpi_total_loss_cost'].median():.2f}")
print(f"  % with zero loss cost: {(df['kpi_total_loss_cost'] == 0).sum() / len(df) * 100:.1f}%")

print(f"\nLoss Ratio Distribution (kpi_actual_loss_ratio):")
print(f"  Mean: {df['kpi_actual_loss_ratio'].mean():.3f}")
print(f"  Median: {df['kpi_actual_loss_ratio'].median():.3f}")
print(f"  % with zero loss ratio: {(df['kpi_actual_loss_ratio'] == 0).sum() / len(df) * 100:.1f}%")

print(f"\n" + "="*80)
if pct_zero_claims > 80 and pct_zero_adequacy < 10:
    print(f"⚠️  POTENTIAL ISSUE:")
    print(f"   {pct_zero_claims:.1f}% have zero claims")
    print(f"   But only {pct_zero_adequacy:.1f}% have adequacy_score = 0")
    print(f"   This suggests Adequacy may not be measuring real loss experience.")
else:
    print(f"✅ Adequacy appears to track claim frequency reasonably well.")

§4.1 BLOCKER — ADEQUACY REALITY CHECK

Physicians with ZERO claims (kpi_total_frequency = 0):
  Count: 159,734
  Percentage: 96.4%

Adequacy Score Distribution:
  Mean: 3.215
  Median: 3.110
  Min: 1.000
  Max: 7.000
  Std Dev: 1.393

Physicians with Adequacy Score = 0:
  Count: 0
  Percentage: 0.0%

Loss Cost Distribution (kpi_total_loss_cost):
  Mean: 5844.75
  Median: 0.00
  % with zero loss cost: 96.4%

Loss Ratio Distribution (kpi_actual_loss_ratio):
  Mean: 0.543
  Median: 0.000
  % with zero loss ratio: 96.4%

⚠️  POTENTIAL ISSUE:
   96.4% have zero claims
   But only 0.0% have adequacy_score = 0
   This suggests Adequacy may not be measuring real loss experience.


## Cell 3: §4.2 BLOCKER — Circularity Check

**Question:** Are specialty/state tiers cut on loss ratio (the outcome)?

**Check:**
- Do high-tier specialties have high loss ratios?
- Or do they measure risk-independent factors?

In [3]:
print("="*80)
print("§4.2 BLOCKER — CIRCULARITY CHECK")
print("="*80)
print()

# Analyze specialty tier
print(f"Specialty Risk Tier Analysis:")
print(f"  Column: score_specialty_risk_tier")
print()

specialty_analysis = df.groupby('score_specialty_risk_tier').agg({
    'kpi_actual_loss_ratio': ['mean', 'median', 'count'],
    'kpi_total_loss_cost': 'mean',
    'composite_score': 'mean'
}).round(3)

print(specialty_analysis)

print(f"\n" + "="*80)
print(f"\nState Venue Risk Analysis:")
print(f"  Column: score_state_venue_risk")
print()

state_analysis = df.groupby('score_state_venue_risk').agg({
    'kpi_actual_loss_ratio': ['mean', 'median', 'count'],
    'composite_score': 'mean'
}).round(3)

print(state_analysis)

print(f"\n" + "="*80)
print(f"\n⚠️  ANALYSIS:")
print(f"If specialty/state tiers show strong correlation with loss ratio,")
print(f"validation against loss ratio could be CIRCULAR (measuring MagMutual's")
print(f"pricing decisions, not true physician risk).")

§4.2 BLOCKER — CIRCULARITY CHECK

Specialty Risk Tier Analysis:
  Column: score_specialty_risk_tier

                          kpi_actual_loss_ratio                \
                                           mean median  count   
score_specialty_risk_tier                                       
1                                         0.109    0.0   3596   
4                                         0.383    0.0  11017   
7                                         0.476    0.0  29238   
11                                        0.460    0.0  23078   
14                                        0.517    0.0  62873   
17                                        0.557    0.0   7376   
20                                        0.854    0.0   7217   
24                                        0.768    0.0  11775   
27                                        0.884    0.0   7306   
30                                        1.175    0.0   2147   

                          kpi_total_loss_cost composi

## Cell 4: §4.3 BLOCKER — Data Quality Check

**Question:** Are there sentinel values (−99) or missing data?

In [4]:
print("="*80)
print("§4.3 BLOCKER — DATA QUALITY CHECK")
print("="*80)
print()

# Check for sentinel values and missing data
print(f"Checking for sentinel values (-99) and missing data...\n")

sentinel_count = 0
null_count = 0

for col in df.columns:
    if df[col].dtype in ['int64', 'float64']:
        sentinels = (df[col] == -99).sum()
        nulls = df[col].isnull().sum()
        
        if sentinels > 0 or nulls > 0:
            pct_sentinel = (sentinels / len(df)) * 100
            pct_null = (nulls / len(df)) * 100
            print(f"{col:40s}: {sentinels:6,} sentinels ({pct_sentinel:5.2f}%) | {nulls:6,} nulls ({pct_null:5.2f}%)")
            sentinel_count += sentinels
            null_count += nulls

print(f"\nTotal sentinel values (-99) found: {sentinel_count:,}")
print(f"Total null values found: {null_count:,}")

if sentinel_count > 0:
    print(f"\n⚠️  WARNING: Sentinel values (-99) found!")
    print(f"   These should be converted to NaN before scoring.")
else:
    print(f"\n✅ No sentinel values (-99) found.")

if null_count > 0:
    print(f"\n⚠️  WARNING: Missing values found!")
    print(f"   Check that these are handled correctly.")
else:
    print(f"✅ No missing values found.")

§4.3 BLOCKER — DATA QUALITY CHECK

Checking for sentinel values (-99) and missing data...

NPI                                     :      0 sentinels ( 0.00%) |  4,598 nulls ( 2.78%)
kpi_indemnity_frequency                 :      0 sentinels ( 0.00%) | 14,139 nulls ( 8.54%)
kpi_total_severity                      :      0 sentinels ( 0.00%) | 159,734 nulls (96.44%)
capacity_score                          :      0 sentinels ( 0.00%) | 60,802 nulls (36.71%)

Total sentinel values (-99) found: 0
Total null values found: 239,273

✅ No sentinel values (-99) found.

⚠️  WARNING: Missing values found!
   Check that these are handled correctly.


## Cell 5: Component Score Distribution

Overview of your four component scores

In [5]:
print("="*80)
print("COMPONENT SCORE DISTRIBUTIONS")
print("="*80)

components = {
    'adequacy_score (40%)': df['adequacy_score'],
    'capacity_score (25%)': df['capacity_score'],
    'appetite_score (25%)': df['appetite_score'],
    'environment_score (10%)': df['environment_score']
}

for name, col in components.items():
    print(f"\n{name}")
    print(f"  Mean: {col.mean():.3f} | Median: {col.median():.3f} | Std: {col.std():.3f}")
    print(f"  Range: [{col.min():.1f}, {col.max():.1f}]")
    print(f"  Value counts:")
    print(col.value_counts().head(10))

print(f"\n\n{'='*80}")
print(f"COMPOSITE SCORE (Final Output)")
print(f"{'='*80}")
print(f"\ncomposite_score distribution:")
print(f"  Mean: {df['composite_score'].mean():.3f}")
print(f"  Median: {df['composite_score'].median():.3f}")
print(f"  Std Dev: {df['composite_score'].std():.3f}")
print(f"  Range: [{df['composite_score'].min():.1f}, {df['composite_score'].max():.1f}]")
print(f"\nValue counts:")
print(df['composite_score'].value_counts().sort_index())

COMPONENT SCORE DISTRIBUTIONS

adequacy_score (40%)
  Mean: 3.215 | Median: 3.110 | Std: 1.393
  Range: [1.0, 7.0]
  Value counts:
adequacy_score
2.241379    4464
2.862069    4167
4.103448    3909
3.482759    3727
1.000000    3550
3.110345    3472
2.986207    3460
3.606897    3459
2.737931    3456
1.248276    3442
Name: count, dtype: int64

capacity_score (25%)
  Mean: 5.730 | Median: 6.276 | Std: 3.047
  Range: [1.0, 10.0]
  Value counts:
capacity_score
1.000000     23718
10.000000     5435
8.758621      5338
6.896552      5303
8.137931      5295
7.517241      5223
5.034483      5123
7.827586      5069
8.448276      4979
9.379310      4910
Name: count, dtype: int64

appetite_score (25%)
  Mean: 5.392 | Median: 5.407 | Std: 1.277
  Range: [1.0, 9.8]
  Value counts:
appetite_score
5.732759    1486
6.586207    1469
5.150862    1371
5.748276    1352
4.103448    1327
4.685345    1285
5.034483    1161
5.500000     806
5.316614     800
5.344828     705
Name: count, dtype: int64

environment_

## Summary

Three critical blockers that would prevent pricing use:

1. **§4.1 Adequacy Reality** — Does 40% measure real loss or burn factors?
2. **§4.2 Circularity** — Are specialty/state tiers cut on the outcome?
3. **§4.3 Data Quality** — Are sentinel values/missing data handled correctly?